# Token Context Buildup

How context accumulates in token representations: accumulation rate,
source attribution, position diversity, and embedding distance.

## Why This Matters

Token context buildup tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_context_buildup import (
    context_accumulation_rate, context_source_attribution,
    position_context_diversity, embedding_distance_tracking,
    context_buildup_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Context Accumulation Rate

How much does the representation change at each layer?

In [ ]:
result = context_accumulation_rate(model, tokens, position=-1)
print(f"Trend: {result['update_trend']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['relative_update'] * 20)
    print(f"  Layer {p['layer']}: update={p['update_norm']:.4f}, "
          f"relative={p['relative_update']:.4f}, cos={p['pre_post_cosine']:.4f} {bar}")

## Context Source Attribution

Is context coming more from attention or MLP?

In [ ]:
result = context_source_attribution(model, tokens, position=-1)
print(f"Mean attn fraction: {result['mean_attn_fraction']:.2%}\n")
for p in result['per_layer']:
    attn_bar = '█' * int(p['attn_fraction'] * 20)
    mlp_bar = '█' * int(p['mlp_fraction'] * 20)
    print(f"  Layer {p['layer']}: attn={p['attn_fraction']:.2f} {attn_bar}")
    print(f"           mlp ={p['mlp_fraction']:.2f} {mlp_bar}")

## Position Context Diversity

Do positions become more distinct through layers?

In [ ]:
for layer in range(4):
    result = position_context_diversity(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_diverse'] else ''
    bar = '█' * int(result['mean_pairwise_similarity'] * 20)
    print(f"  Layer {layer}: sim={result['mean_pairwise_similarity']:.4f} {bar}{flag}")

## Embedding Distance Tracking

How far does the representation drift from the initial embedding?

In [ ]:
result = embedding_distance_tracking(model, tokens, position=-1)
print(f"Final distance: {result['final_distance']:.4f}")
print(f"Final cosine: {result['final_cosine']:.4f}\n")
for p in result['per_layer']:
    bar = '█' * int(p['distance_from_embed'] * 5)
    print(f"  Layer {p['layer']}: dist={p['distance_from_embed']:.4f}, "
          f"cos={p['cosine_to_embed']:.4f} {bar}")

## Context Buildup Summary

Combined context buildup analysis.

In [ ]:
result = context_buildup_summary(model, tokens, position=-1)
print(f"Trend: {result['update_trend']}")
print(f"Final distance: {result['final_distance']:.4f}")
print(f"Final cosine: {result['final_cosine']:.4f}")
print(f"Mean attn fraction: {result['mean_attn_fraction']:.2%}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: update={p['relative_update']:.4f}, "
          f"attn={p['attn_fraction']:.2f}, "
          f"dist={p['distance_from_embed']:.4f}, cos={p['cosine_to_embed']:.4f}")